# 02 — Join datasets on `job_link`

**Input:** cleaned tables from `01_data_overview_and_cleaning.ipynb`  
**Output:** one analysis-ready dataset (postings + skills; summary optional)

Join rule:
- key = `job_link`
- expected cardinality ≈ **1 : 1 : 1** (postings ↔ skills ↔ summary)
- default join = **left** from cleaned postings (keep all jobs; skills/summary may be missing)

> Summary text is huge (~4.8GB raw). By default we join **postings + skills only**, and attach summary only if you flip a flag / provide a cleaned summary file.


## 0. Setup


In [ ]:
from pathlib import Path
import warnings

import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)

# Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/Arusto_LinkedIn')

DATA_DIR = Path(".")
CLEAN_DIR = DATA_DIR / "cleaned"
OUT_DIR = CLEAN_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- join options --------------------------------------------------------
INCLUDE_SUMMARY = False   # True only if you have cleaned summary ready
JOIN_HOW = "left"         # left = keep all postings; inner = only jobs with skills

print("CLEAN_DIR:", CLEAN_DIR.resolve())
print("INCLUDE_SUMMARY:", INCLUDE_SUMMARY)
print("JOIN_HOW:", JOIN_HOW)


In [ ]:
def read_clean(name):
    """Load cleaned parquet, else csv.gz."""
    pq = CLEAN_DIR / f"{name}.parquet"
    gz = CLEAN_DIR / f"{name}.csv.gz"
    if pq.exists():
        print(f"loading {pq.name}")
        return pd.read_parquet(pq)
    if gz.exists():
        print(f"loading {gz.name}")
        return pd.read_csv(gz)
    raise FileNotFoundError(
        f"Missing {name}.parquet / {name}.csv.gz in {CLEAN_DIR}. "
        "Run notebook 01 first."
    )


def save_df(df, name):
    pq = OUT_DIR / f"{name}.parquet"
    gz = OUT_DIR / f"{name}.csv.gz"
    try:
        df.to_parquet(pq, index=False)
        print(f"saved {pq.name}  ({len(df):,} rows, {pq.stat().st_size/1e6:.1f} MB)")
    except Exception as e:
        print(f"parquet failed ({e}); writing csv.gz")
        df.to_csv(gz, index=False, compression="gzip")
        print(f"saved {gz.name}  ({len(df):,} rows, {gz.stat().st_size/1e6:.1f} MB)")


## 1. Load cleaned tables


In [ ]:
postings = read_clean("postings_clean")
skills = read_clean("skills_clean")

print("postings:", postings.shape)
print("skills  :", skills.shape)
print()
print("postings cols:", list(postings.columns))
print("skills cols  :", list(skills.columns))


In [ ]:
# quick key sanity before join
assert postings["job_link"].is_unique, "postings job_link not unique"
assert skills["job_link"].is_unique, "skills job_link not unique"

print("postings unique links:", postings["job_link"].nunique())
print("skills unique links  :", skills["job_link"].nunique())
print(
    "overlap:",
    len(set(postings["job_link"]) & set(skills["job_link"])),
)


## 2. Join postings + skills

We keep skills as **one row per job** (comma / pipe string + `n_skills`), not the exploded long form.  
That gives a single wide-ish job-level dataset — better default for dashboard filters and later scoring.


In [ ]:
# only bring skill columns we need (avoid colliding names)
skill_cols = ["job_link", "job_skills", "n_skills"]
if "skill_list_pipe" in skills.columns:
    skill_cols.append("skill_list_pipe")

skills_for_join = skills[skill_cols].copy()

jobs = postings.merge(skills_for_join, on="job_link", how=JOIN_HOW)

jobs["has_skills"] = jobs["n_skills"].notna()

print("joined shape:", jobs.shape)
print("has_skills rate:", f"{jobs['has_skills'].mean():.1%}")
print()
print("rows by has_skills:")
print(jobs["has_skills"].value_counts())
jobs.head(3)


## 3. Optional: attach summary

Skip unless `INCLUDE_SUMMARY=True` and you already have `summary_clean` (full) or are okay using the sample only for testing.


In [ ]:
if INCLUDE_SUMMARY:
    # prefer full cleaned summary if present; else sample (for smoke test only)
    summary_name = None
    for candidate in ["summary_clean", "summary_clean_sample"]:
        if (CLEAN_DIR / f"{candidate}.parquet").exists() or (CLEAN_DIR / f"{candidate}.csv.gz").exists():
            summary_name = candidate
            break

    if summary_name is None:
        raise FileNotFoundError("No summary_clean / summary_clean_sample found in cleaned/")

    summary = read_clean(summary_name)
    assert summary["job_link"].is_unique, "summary job_link not unique"

    summary_for_join = summary[["job_link", "job_summary"]].copy()
    if "summary_len" in summary.columns:
        summary_for_join["summary_len"] = summary["summary_len"]

    before_cols = jobs.shape[1]
    jobs = jobs.merge(summary_for_join, on="job_link", how=JOIN_HOW)
    jobs["has_summary"] = jobs["job_summary"].notna()

    print(f"attached from: {summary_name}")
    print("shape after summary join:", jobs.shape, f"(+{jobs.shape[1] - before_cols} cols)")
    print("has_summary rate:", f"{jobs['has_summary'].mean():.1%}")
else:
    print("summary not attached (INCLUDE_SUMMARY=False)")
    # still useful flag from postings cleaning, if present
    if "got_summary" in jobs.columns:
        print("proxy got_summary True rate:", f"{jobs['got_summary'].mean():.1%}")


## 4. Light post-join checks


In [ ]:
print("final shape:", jobs.shape)
print("duplicate job_link after join:", jobs["job_link"].duplicated().sum())
print()

# coverage by country — handy for the demand story later
if "search_country" in jobs.columns:
    cov = (
        jobs.groupby("search_country", dropna=False)
        .agg(
            n_jobs=("job_link", "size"),
            with_skills=("has_skills", "sum"),
        )
        .assign(skills_rate=lambda d: (d["with_skills"] / d["n_jobs"]).round(3))
        .sort_values("n_jobs", ascending=False)
    )
    display(cov)

print()
print("missingness (top):")
miss = jobs.isna().mean().sort_values(ascending=False).head(12)
print((miss * 100).round(2).astype(str) + "%")


## 5. Save the joined dataset

Two exports:
- `jobs_joined` — full joined table
- `jobs_joined_us` — US-only slice (faster for next EDA / dashboard prototyping)


In [ ]:
save_df(jobs, "jobs_joined")

if "search_country" in jobs.columns:
    jobs_us = jobs.loc[jobs["search_country"] == "United States"].copy()
    save_df(jobs_us, "jobs_joined_us")
    print(f"\nUS slice: {len(jobs_us):,} / {len(jobs):,} rows")
else:
    print("no search_country column; skipped US slice")

print("\nfiles now in", OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob("jobs_joined*")):
    print(f"  - {p.name:28} {p.stat().st_size/1e6:7.1f} MB")


### Done

You now have **one job-level dataset**.  

Next natural step: EDA on `jobs_joined_us` (title patterns, top skills from `skills_long` or `skill_list_pipe`, role × skill demand) → course opportunity score → dashboard.
